In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# SET WORKING DIRECTORY
# ------------------------------------------------------------
data_dir = '/Users/nuha/Desktop/Xenium/20251217_Xenium/Tissue_0038728'
os.chdir(data_dir)

# ------------------------------------------------------------
# CREATE OUTPUT FOLDER
# ------------------------------------------------------------
output_dir = os.path.join(data_dir, "Generated_Images")
os.makedirs(output_dir, exist_ok=True)

print("Images will be saved to:")
print(output_dir)

# ------------------------------------------------------------
# LOAD FILES (ONLY ONCE → FAST)
# ------------------------------------------------------------
df_reads_UMIs = pd.read_csv("Clean_UMIs.csv")
df_position_cells = pd.read_csv("Clean_cells.csv")
df_UMAP = pd.read_csv("projection.csv")

df_reads_UMIs.columns = ['Num','Cell_code','ENSG','Gene','Type','UMI']

# ------------------------------------------------------------
# GENES TO PROCESS
# ------------------------------------------------------------
GENE_LIST = [
    "PDGFRA",
]

# ------------------------------------------------------------
# GAMMA SETTING
# ------------------------------------------------------------
GAMMA = 0.5   # < 1 boosts low expression

# ------------------------------------------------------------
# COORDINATES (CALCULATE ONCE)
# ------------------------------------------------------------
cell_codes = df_position_cells.iloc[:,1]

cell_positions_x = df_position_cells.iloc[:,2].to_numpy()
cell_positions_y = df_position_cells.iloc[:,3].to_numpy()

UMAP_1 = df_UMAP.iloc[:,1].to_numpy()
UMAP_2 = df_UMAP.iloc[:,2].to_numpy()

# ------------------------------------------------------------
# RELATIVE INTENSITY + GAMMA CORRECTION
# ------------------------------------------------------------
def relative_intensity_gamma(values, gamma=1.0, low_pct=2, high_pct=98):

    values = np.array(values, dtype=float)

    # Percentile clipping
    vmin = np.percentile(values, low_pct)
    vmax = np.percentile(values, high_pct)
    clipped = np.clip(values, vmin, vmax)

    # Normalize to [0,1]
    rel = (clipped - clipped.min()) / (clipped.max() - clipped.min() + 1e-9)

    # Gamma correction
    rel_gamma = np.power(rel, gamma)

    return rel_gamma

# ------------------------------------------------------------
# SAVE FUNCTION
# ------------------------------------------------------------
def save_plot(fig, filename):
    filepath = os.path.join(output_dir, filename)
    fig.savefig(filepath, dpi=600)
    print("Saved:", filepath)
    plt.close(fig)

# ------------------------------------------------------------
# PRE-GROUP ALL GENES (FAST)
# ------------------------------------------------------------
gene_grouped = (
    df_reads_UMIs
    .groupby(['Gene', 'Cell_code'])['UMI']
    .sum()
)

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
for gene in GENE_LIST:

    print(f"\nProcessing {gene}")

    try:
        gene_counts = gene_grouped.loc[gene]
    except KeyError:
        print("Gene not found, skipping")
        continue

    UMI_values = (
        cell_codes
        .map(gene_counts)
        .fillna(0)
        .to_numpy()
    )

    rel_UMI = relative_intensity_gamma(
        UMI_values,
        gamma=GAMMA
    )

    # ----------------------------------------
    # UMAP PLOT
    # ----------------------------------------
    fig, ax = plt.subplots(figsize=(6,6))

    ax.scatter(
        UMAP_1,
        UMAP_2,
        c=rel_UMI,
        s=0.3,
        alpha=0.3,
        cmap=plt.cm.plasma,
        vmin=0,
        vmax=1
    )

    ax.set_xlabel("UMAP_1")
    ax.set_ylabel("UMAP_2")

    save_plot(fig, f"{gene}_UMAP_Relative_Gamma{GAMMA}_Plasma.png")

    # ----------------------------------------
    # TISSUE PLOT
    # ----------------------------------------
    fig, ax = plt.subplots(figsize=(8,8))

    ax.scatter(
        cell_positions_x,
        cell_positions_y,
        c=rel_UMI,
        s=0.3,
        alpha=0.3,
        cmap=plt.cm.plasma,
        vmin=0,
        vmax=1
    )

    ax.set_xlabel("Pos_x")
    ax.set_ylabel("Pos_y")

    ax.set_aspect('equal')

    try:
        ax.set_box_aspect(1)
    except:
        pass

    save_plot(fig, f"{gene}_Tissue_Relative_Gamma{GAMMA}_Plasma.png")

print("\nAll genes processed successfully.")

Images will be saved to:
/Users/nuha/Desktop/Xenium/20251217_Xenium/Tissue_0038728/Generated_Images

Processing PDGFRA
Saved: /Users/nuha/Desktop/Xenium/20251217_Xenium/Tissue_0038728/Generated_Images/PDGFRA_UMAP_Relative_Gamma0.5_Plasma.png
Saved: /Users/nuha/Desktop/Xenium/20251217_Xenium/Tissue_0038728/Generated_Images/PDGFRA_Tissue_Relative_Gamma0.5_Plasma.png

All genes processed successfully.
